## Settings

In [51]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [52]:
## libraries
import sys
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.decomposing import (
    train_decomposed_separability,
    train_decomposed_exhaustiveness,
    compile_decomposed_separability,
    compile_decomposed_exhaustiveness,
    stat_decomposed_separability,
    stat_decomposed_exhaustiveness,
    stat_decomposed_sufficiency,
    stat_decomposed_additive,
)

## constants
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET,
)

## Initialization

In [53]:
## reproducibility
RANDOM_STATE = 42
N_REPEATS = 30

## load data and models
data = load_processed_data()
models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

## view model surface
n_mods = len(models)
print(f"Learned Models: {n_mods} estimators")

Original Data: 32 features, 25 observations
Learned Models: 9 estimators


## Training

In [54]:
## train decomposed separability
cached_decomposed_separability = globals().get("results_dict_decomposed_separability")
if (
    not isinstance(cached_decomposed_separability, dict)
    or cached_decomposed_separability.get("n_repeats") != N_REPEATS
    or cached_decomposed_separability.get("random_state") != RANDOM_STATE
):
    results_dict_decomposed_separability = train_decomposed_separability(
        data = data,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
        n_jobs = -1,
    )

In [55]:
## train decomposed exhaustiveness
cached_decomposed_exhaustiveness = globals().get("results_dict_decomposed_exhaustiveness")
if (
    not isinstance(cached_decomposed_exhaustiveness, dict)
    or cached_decomposed_exhaustiveness.get("n_repeats") != N_REPEATS
    or cached_decomposed_exhaustiveness.get("random_state") != RANDOM_STATE
):
    results_dict_decomposed_exhaustiveness = train_decomposed_exhaustiveness(
        data = data,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
        n_jobs = -1,
    )

## Post-Processing

In [56]:
## compile decomposed separability results
results_decomposed_separability, predictions_decomposed_separability = compile_decomposed_separability(
    results = results_dict_decomposed_separability
)

## compile decomposed exhaustiveness results
results_decomposed_exhaustiveness, predictions_decomposed_exhaustiveness = compile_decomposed_exhaustiveness(
    results = results_dict_decomposed_exhaustiveness
)

## Additive Separability Test
The additive separability test probes whether the canonical specification $y^* = C(X) + R(Z)$ matches the EI transfer performance of richer alternatives. Each model is refit under additive, interaction, interaction-joint, joint, capacity-only, and dynamics-only specifications under LOGO-CV (domain), with predictions averaged across repeated seeds before scoring EI per (model x domain).

- **Additive**: $y^* = C(X) + R(Z)$ with separate capacity and residual stages.
- **Interaction**: Additive with $X \otimes Z$ cross-terms appended to the residual stage.
- **Interaction-Joint**: Single-stage learner over $[X, Z, X \otimes Z]$.
- **Joint**: Single-stage learner over $[X, Z]$ with no factorization.
- **Capacity-Only**: $y^* = C(X)$, dropping the residual stage.
- **Dynamics-Only**: $y^* = R(Z)$, dropping the capacity stage.

### Reporting Convention
Each specification contributes one repeated-seed-averaged value per (model $\times$ domain). The summary table reports the mean across (model $\times$ domain) for each frontier metric. `EI` is the primary criterion; `VR`, `MV`, and `MS` are reported alongside as auxiliary frontier diagnostics. Higher `EI` indicates stronger transfer performance; lower `VR`, `MV`, and `MS` indicate fewer and smaller frontier violations.

In [64]:
## mean frontier metrics by specification
results_separability = stat_decomposed_separability(
    results = results_decomposed_separability,
    decimals = 3
)

display(results_separability)

,EI,VR,MV,MS
SPECIFICATION,,,,
Additive,0.767,0.087,3.829,5.520
Interaction,0.747,0.058,0.303,1377.226
Interaction Joint,0.763,0.109,0.479,5.287
Joint,0.768,0.113,0.616,5.046
Capacity Only,0.755,0.102,3.960,5.562
Dynamics Only,0.776,0.120,0.511,4.717


_The additive specification lies in the same narrow EI band as the relaxed joint and dynamics-only alternatives, while the interaction and capacity-only variants are lower. No richer specification produces a meaningfully larger EI gain than the additive baseline, and the auxiliary frontier diagnostics remain broadly comparable across the main decomposed specifications. This supports the additive $C(X) + R(Z)$ form as a parsimonious representation of the observed frontier._

## Exhaustiveness Test
The exhaustiveness test probes whether the capacity stage $C(X)$ fully absorbs the contribution of $X$ to $y^*$. Under true additivity $y^* = C(X) + R(Z)$, the slack $s = y^* - \hat{C}(X)$ should depend on dynamics rather than topology. Within each LOGO (domain) fold, separate learners are fit for $X \to s$ and $Z \to s$, repeated across seeds, and their seed-averaged domain-level mean absolute errors are paired on (model $\times$ domain).

- **$H_0$**: The slack is at least as predictable from $X$ as from $Z$ ($\Delta |\mathrm{error}| \le 0$).
- **$H_1$**: The slack is more predictable from $Z$ than from $X$ ($\Delta |\mathrm{error}| > 0$).

Rejecting $H_0$ supports that $C(X)$ exhausts topology's contribution to $y^*$ and that the residual structure is governed by dynamics.

### Empirical Test Specification
Repeated-seed-averaged predictions for $X \to s$ and $Z \to s$ are aggregated to (model $\times$ domain) by mean absolute error, then differenced as $\Delta = |\mathrm{error}|_{X \to s} - |\mathrm{error}|_{Z \to s}$. The paired differences are tested with a one-sided Wilcoxon signed-rank statistic ($H_1: \Delta > 0$). Effect size is reported as the rank-biserial correlation derived from the signed-rank statistic.

In [61]:
## one-sided wilcoxon test for decomposition exhaustiveness
results_exhaustiveness, summary_exhaustiveness = stat_decomposed_exhaustiveness(
    predictions = predictions_decomposed_exhaustiveness,
    decimals = 2
)

display(results_exhaustiveness)
display(summary_exhaustiveness)

Paired One-Sided Test (Wilcoxon Signed-Rank): n = 45
H0: Δ |ERROR| <= 0
H1: Δ |ERROR| > 0
Δ |ERROR|: paired X -> slack error minus Z -> slack error
Rank-biserial r: positive values favor Z -> slack
*** p < 0.001, ** p < 0.01, * p < 0.05


,N,Z BETTER,X BETTER,MEAN Δ |ERROR|,MEDIAN Δ |ERROR|,RANK-BISERIAL R,WILCOXON P,SIG.,DIFF.
0,45,32,13,268.34,0.65,0.52,0.0,**,Yes


,MEAN |ERROR|,STD |ERROR|,N
RESIDUAL FEATURES,,,
X -> SLACK,278.13,1633.82,45
Z -> SLACK,9.80,18.03,45


_The slack left behind by $\hat{C}(X)$ is better predicted from dynamics ($Z$) than from topology ($X$) in a majority of paired domain-level comparisons, with a positive median error shift and a significant one-sided Wilcoxon result. This supports the interpretation that $C(X)$ absorbs much of topology's contribution to $y^*$, leaving residual variation that is more naturally explained by dynamics._

## Sufficiency Non-Inferiority Test
The sufficiency test probes whether the additive specification $y^* = C(X) + R(Z)$ is non-inferior to richer alternatives within a prespecified EI margin $\delta$. Each relaxed specification is paired against additive on repeated-seed-averaged (model $\times$ domain) EI values and tested against the equivalence margin.

- **$H_0$**: The relaxed specification exceeds additive by at least $\delta$ ($\Delta \mathrm{EI} \ge \delta$).
- **$H_1$**: The relaxed specification does not exceed additive by $\delta$ ($\Delta \mathrm{EI} < \delta$).

Rejecting $H_0$ supports that the relaxed specification fails to deliver an EI gain larger than $\delta$ over additive, i.e. additive is non-inferior at margin $\delta$.

### Empirical Test Specification
The non-inferiority margin is fixed at $\delta = 0.05$ EI points, anchored to the precision typical of frontier evaluation. Paired gaps $\Delta \mathrm{EI} = \mathrm{EI}_{\text{spec}} - \mathrm{EI}_{\text{additive}}$ are tested against $\delta$ with three statistics: a one-sample $t$-test against the margin, a one-sided Wilcoxon signed-rank test against the margin, and a sign test against the margin. The Wilcoxon $p$-value is the primary statistic; the $t$-test and sign test are reported as parametric and assumption-free robustness checks.

In [59]:
## non-inferiority tests for additive sufficiency
NI_DELTA = 0.05
results_sufficiency = stat_decomposed_sufficiency(
    results = results_decomposed_separability,
    delta = NI_DELTA,
    decimals = 4
)

display(results_sufficiency)

Paired Non-Inferiority Test (Wilcoxon Signed-Rank): n = 45, δ = 0.05
H0: Δ EI >= δ
H1: Δ EI < δ
Median Δ EI: relaxed specification minus additive
Holm-adj. p: Holm-Bonferroni adjusted Wilcoxon p-value
NI.: Yes if Holm-adj. p < 0.05 and Median Δ EI < δ
*** p < 0.001, ** p < 0.01, * p < 0.05


,N,SPEC BETTER,ADDITIVE BETTER,MEAN Δ EI,MEDIAN Δ EI,T-TEST P,WILCOXON P,SIGN P,HOLM-ADJ. P,SIG.,NI.
SPECIFICATION,,,,,,,,,,,
Interaction,45,24,21,-0.0199,0.0016,0.0029,0.0,0.0,0.0,***,Yes
Interaction Joint,45,16,29,-0.0039,-0.0137,0.0023,0.0,0.0,0.0,***,Yes
Joint,45,16,29,0.0006,-0.0100,0.0026,0.0,0.0,0.0,***,Yes
Capacity Only,45,13,32,-0.0123,-0.0139,0.0000,0.0,0.0,0.0,***,Yes
Dynamics Only,45,19,26,0.0093,-0.0036,0.0102,0.0,0.0,0.0,***,Yes


In [60]:
## additive frontier quality summary
results_additive = stat_decomposed_additive(
    results = results_decomposed_separability,
    sufficiency = results_sufficiency,
    delta = NI_DELTA,
    decimals = 4
)

display(results_additive)

,MEAN EI,SD EI,MAX MEAN Δ EI,MARGIN Δ,NON-INFERIOR,WORST ADJ. P
0,0.767,0.0562,0.0093,0.05,5/5,0.0


_The additive specification is non-inferior to every relaxed alternative within the prespecified margin $\delta = 0.05$, with paired EI gaps centered near zero and Holm-adjusted Wilcoxon tests supporting non-inferiority across specifications. Together with the exhaustiveness result, this favors the canonical $C(X) + R(Z)$ decomposition: richer interaction or joint learners do not provide a practically meaningful EI gain, while residual variation after $\hat{C}(X)$ is better explained by dynamics than topology._